# Audiobook workflow with ElevenLabs TTS

This notebook generates story text using the existing OpenAI story helper and converts the narration to audio using the ElevenLabs text-to-speech API.

In [1]:
import sys
from pathlib import Path
import os
import requests
from dotenv import load_dotenv
from IPython.display import Audio, display
from pprint import pprint

root_dir = Path.cwd().parent
sys.path.append(str(root_dir))

from whisper_client import generate_story_text

load_dotenv()

ELEVENLABS_API_KEY = os.getenv("ELEVENLABS_API_KEY")
ELEVENLABS_VOICE_ID = os.getenv("ELEVENLABS_VOICE_ID")

if ELEVENLABS_API_KEY is None:
    raise EnvironmentError("ELEVENLABS_API_KEY must be set in the environment.")
if ELEVENLABS_VOICE_ID is None:
    raise EnvironmentError("ELEVENLABS_VOICE_ID must be set in the environment.")

headers = {
    "xi-api-key": ELEVENLABS_API_KEY,
    "Content-Type": "application/json",
}

project_root = root_dir
prd_file = project_root / "audiobook_prd.md"
preview_audio_file = project_root / "output" / "sample_story_preview_elevenlabs.mp3"
full_audio_file = project_root / "output" / "full_story_elevenlabs.mp3"

print(f"Using PRD file: {prd_file}")
print(f"Preview audio file: {preview_audio_file}")
print(f"Full audio file: {full_audio_file}")

Using PRD file: /Users/parikshitmehta/Documents/audiobook_project/audiobook_prd.md
Preview audio file: /Users/parikshitmehta/Documents/audiobook_project/output/sample_story_preview_elevenlabs.mp3
Full audio file: /Users/parikshitmehta/Documents/audiobook_project/output/full_story_elevenlabs.mp3


## Test voice with a custom sentence

Before generating the full audiobook, test the configured voice to ensure it sounds good. Edit the test sentence below and run the cell.


In [8]:
# Edit this test sentence to your liking
test_sentence = "Once upon a time, in the cozy month of December, Mehta family was getting ready for a trip to India."

# Synthesize the test sentence
test_audio_path = Path(project_root) / "output" / "test_voice_elevenlabs.mp3"

try:
    from whisper_client import synthesize_with_elevenlabs
    
    synthesize_with_elevenlabs(
        text=test_sentence,
        output_file_path=str(test_audio_path),
        voice_id=ELEVENLABS_VOICE_ID,
        model_id="eleven_v3"
    )
    print(f"✓ Test audio generated successfully")
    print(f"Test sentence: {test_sentence}")
    print(f"Saved to: {test_audio_path}")
    display(Audio(filename=str(test_audio_path)))
except Exception as e:
    print(f"✗ Error generating test audio: {e}")


✓ Test audio generated successfully
Test sentence: Once upon a time, in the cozy month of December, Mehta family was getting ready for a trip to India.
Saved to: /Users/parikshitmehta/Documents/audiobook_project/output/test_voice_elevenlabs.mp3


## Story prompt

Define the story details and tone for the audiobook. This prompt is used to generate a child-friendly narrative in the next cell.

In [ ]:
story_prompt = (
    "Meera is going on a vacation trip to India with her family. "
    "She visits grandparents, cousins, and explores colorful markets, food, and parks. "
    "Write a warm, calm, and simple audiobook story for a 4-year-old, with gentle narration and playful moments. "
    "Keep the language easy and the story soothing for young listeners."
)

print(story_prompt)

## Generate story text only

Run this cell to generate the story text from the PRD and prompt. No audio is created yet.

In [ ]:
story_text = generate_story_text(
    prompt=story_prompt,
    prd_file_path=str(prd_file),
    model="gpt-4o-mini",
    temperature=0.8,
    max_output_tokens=1000,
)

print('--- Generated story text ---')
pprint(story_text)

## Review and approve story text

Read the generated story text. If the story looks good, run the preview audio cell below.

In [ ]:
from whisper_client import synthesize_with_elevenlabs

preview_sentences = story_text.split('.')[:30]
preview_text = '.'.join(s.strip() for s in preview_sentences if s).strip()
if not preview_text.endswith('.'):
    preview_text += '.'

print('--- Story preview text ---')
pprint(preview_text)

preview_audio_path = synthesize_with_elevenlabs(
    text=preview_text,
    output_file_path=str(preview_audio_file),
    voice_id=ELEVENLABS_VOICE_ID,
)

print(f'Generated preview audio: {preview_audio_path}')
display(Audio(filename=str(preview_audio_path)))


## Listen to the preview

If the preview narration sounds good, run the final cell to synthesize the full story audio.

In [ ]:
from whisper_client import synthesize_with_elevenlabs

full_audio_path = synthesize_with_elevenlabs(
    text=story_text,
    output_file_path=str(full_audio_file),
    voice_id=ELEVENLABS_VOICE_ID,
)

print(f'Generated full story audio: {full_audio_path}')
display(Audio(filename=str(full_audio_path)))
